# Basic Model

In [10]:
# libraries import
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

### Getting the data

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from data_loaders import Training_set

In [2]:
ts = Training_set()
x_train, y_train, x_test, y_test= ts.split_small_data()

**quick check**

In [3]:
print(f'{x_train.shape} | {y_train.shape} | {x_test.shape} | {y_test.shape}')

(1461, 3243) | (1461, 2352) | (354, 3243) | (354, 2352)


### Building the model

In [8]:
# check for missing values
print('==== Missing values check ====')
print(f'{sum(x_train.isna().sum()) == 0}')
print(f'{sum(y_train.isna().sum()) == 0}')
print(f'{sum(x_test.isna().sum()) == 0}')
print(f'{sum(y_test.isna().sum()) == 0}')
print('==============================')

==== Missing values check ====
True
True
True
True


In [11]:
def extract_positive_probs(proba_list, y_train):
    """
    Converte la lista restituita da RandomForestClassifier.predict_proba()
    in un array 2D di forma (n_samples, n_targets) con le probabilità della classe 1.
    Gestisce anche il caso limite in cui un'unità sia sempre 0 o sempre 1 nel train set.
    """
    n_samples = proba_list[0].shape[0]
    n_targets = len(proba_list)
    probs_2d = np.zeros((n_samples, n_targets), dtype=np.float32)

    for i, p in enumerate(proba_list):
        if p.shape[1] == 2:
            probs_2d[:, i] = p[:, 1]
        else:
            # Se nel training set l'unità aveva una sola classe (tutti 0 o tutti 1)
            only_class = y_train.iloc[:, i].iloc[0]
            probs_2d[:, i] = float(only_class)
            
    return probs_2d

In [12]:
# Definizione del modello Random Forest Base
# Nota: max_depth e min_samples_leaf evitano che i 2352 output saturino la RAM
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,           # Limita la profondità per velocità e memoria
    min_samples_leaf=5,
    n_jobs=-1,              # Usa tutti i core della CPU
    random_state=42,
    verbose=1
)

# Addestramento
print("\nInizio training Random Forest multi-output...")
rf_model.fit(x_train, y_train)

# Predizione delle probabilità (valori continui tra 0 e 1)
print("\nCalcolo predizioni sul test set...")
raw_preds_test = rf_model.predict_proba(x_test)
y_pred_proba = extract_positive_probs(raw_preds_test, y_train)

# Valutazione con la metrica ufficiale del task (Micro ROC-AUC)
micro_auc = roc_auc_score(y_test.to_numpy(), y_pred_proba, average="micro")
print(f"\n---> Test ROC-AUC (micro average): {micro_auc:.4f} <---")

# Creazione DataFrame con le predizioni formattato come y_test
predictions_df = pd.DataFrame(
    y_pred_proba, 
    index=y_test.index, 
    columns=y_test.columns
)
print("\nPrime 5 righe delle probabilità predette:")
print(predictions_df.head())


Inizio training Random Forest multi-output...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   26.5s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  1.1min finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.



Calcolo predizioni sul test set...


[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s



---> Test ROC-AUC (micro average): 0.7823 <---

Prime 5 righe delle probabilità predette:
                           result_committed_hogga_g1_t0  \
starttime                                                 
2019-01-12 00:00:00+00:00                      0.541058   
2019-01-13 00:00:00+00:00                      0.540478   
2019-01-14 00:00:00+00:00                      0.516514   
2019-01-15 00:00:00+00:00                      0.569023   
2019-01-16 00:00:00+00:00                      0.589854   

                           result_committed_hogga_g1_t1  \
starttime                                                 
2019-01-12 00:00:00+00:00                      0.541058   
2019-01-13 00:00:00+00:00                      0.540478   
2019-01-14 00:00:00+00:00                      0.516514   
2019-01-15 00:00:00+00:00                      0.569023   
2019-01-16 00:00:00+00:00                      0.589854   

                           result_committed_hogga_g1_t2  \
starttime             

[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.9s finished
